# CÓDIGO: MODELAMIENTO POR NAIVE BAYES

## 1. Instalación de librerías

In [1]:
import pandas as pd
import numpy as np
import re
import nltk
import spacy
import warnings
warnings.filterwarnings("ignore")

from nltk.corpus import stopwords
from sklearn.naive_bayes import MultinomialNB, ComplementNB, BernoulliNB
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    recall_score,
    f1_score,
    precision_score,
)
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV

In [2]:
# Descargar recursos de NLTK
nltk.download("stopwords")
nltk.download("punkt")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\dacma\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\dacma\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [16]:
# Cargar modelo de spaCy en español
def cargar_modelo_spacy_es():
    for model_name in ("es_core_news_md", "es_core_news_sm"):
        try:
            print(f"Cargando modelo spaCy: {model_name}")
            return spacy.load(model_name)
        except OSError:
            continue

    print("No se encontro un modelo de spaCy en espanol instalado. Se usara un pipeline basico.")
    return spacy.blank("es")


nlp = cargar_modelo_spacy_es()

print("✅ Librerías cargadas correctamente.")
SEED = 42
np.random.seed(SEED)

Cargando modelo spaCy: es_core_news_md
✅ Librerías cargadas correctamente.


## 2. CARGA DE DATOS

In [4]:

df = pd.read_excel("data/Data_arreglada.xlsx", sheet_name="Sheet1")

In [7]:
# Renombrar columnas para facilitar el trabajo
df.columns = ["Categoria", "Resuelta", "Consulta"]

print(f"📦 Shape original: {df.shape}")
print(df.head(5))

📦 Shape original: (35168, 3)
              Categoria Resuelta  \
0         Carnetización       No   
1  Calendario académico       No   
4         Carnetización       No   
7  Calendario académico       No   
8  Calendario académico       No   

                                            Consulta  
0  Ayuda no se como inscribir asignaturas y nadie...  
1           Donde puedo ver el calendario académico?  
4                              Quiero mi carné nuevo  
7                        Puedo tener mi recibo antes  
8          No tengo cita para inscribir asignaturas.  


## 3. LIMPIEZA INICIAL 

In [8]:
# Eliminar filas donde Consulta o Categoria sean nulas
df = df.dropna(subset=["Consulta", "Categoria"])

In [9]:
# Eliminar filas que no fueron resueltas (Resuelta = 'Sí')
df = df[df["Resuelta"].str.strip().str.lower() != "sí"]
print(f"\n📦 Shape después de eliminar resueltas: {df.shape}")


📦 Shape después de eliminar resueltas: (35168, 3)


In [10]:
# Eliminar filas con consulta vacía o solo espacios
df = df[df["Consulta"].str.strip() != ""]
df = df.reset_index(drop=True)

In [11]:
print(f" Shape final del dataset: {df.shape}")
print(df.head(15))

 Shape final del dataset: (35168, 3)
                                            Categoria Resuelta  \
0                                       Carnetización       No   
1                                Calendario académico       No   
2                                       Carnetización       No   
3                                Calendario académico       No   
4                                Calendario académico       No   
5           ​ Gestión Económica - Información general       No   
6                                Calendario académico       No   
7           ​ Gestión Económica - Información general       No   
8                                Calendario académico       No   
9                                       Carnetización       No   
10                               Calendario académico       No   
11                                      Carnetización       No   
12          ​ Gestión Económica - Información general       No   
13  ​ Gestión Económica - Inconsistenci

## 4. MAPEO DE CATEGORÍAS VÁLIDAS

In [12]:
CATEGORIAS_VALIDAS = [
    "Carnetización",
    "Actualización de datos personales",
    "Calendario académico",
    "Certificados",
    "Gestión Académica",
    "Gestión Económica",
    "Reubicación socioeconómica en Pregrado",
    "Aplazamiento de matrícula inicial",
    "Política de gratuidad (matrícula cero) Pregrado",
    "Información general sobre servicios estudiantiles",
]

def mapear_categoria(categoria: str) -> str:
    """
    Mapea cada categoría original del dataset a una de las CATEGORIAS_VALIDAS.
    Usa coincidencia parcial por palabras clave.
    """
    categoria = str(categoria).strip()

    # Mapeos directos y por palabras clave
    mapeo = {
        "Carnetización":                                    ["carnetización", "carnet", "carné"],
        "Actualización de datos personales":                ["actualización de datos", "datos personales", "actualización"],
        "Calendario académico":                             ["calendario académico", "calendario"],
        "Certificados":                                     ["certificado"],
        "Gestión Académica":                                ["gestión académica", "inscripción", "adiciones",
                                                             "cancelaciones", "asignaturas", "sobrecupo",
                                                             "historia académica", "bloqueo", "grado",
                                                             "homologación", "traslado", "aplazamiento",
                                                             "reingreso", "notas", "prueba", "inglés",
                                                             "doble titulación", "posgrado"],
        "Gestión Económica":                                ["gestión económica", "recibo", "pago",
                                                             "fraccionamiento", "unificación", "devolución",
                                                             "financiación", "matrícula", "pbm",
                                                             "descuento", "electoral", "generación e",
                                                             "icetex", "ser pilo", "exención",
                                                             "reexpedición", "cobro", "deuda"],
        "Reubicación socioeconómica en Pregrado":           ["reubicación socioeconómica", "reubicación",
                                                             "socioeconómica", "socioeconómico"],
        "Aplazamiento de matrícula inicial":                ["aplazamiento de matrícula", "aplazamiento inicial",
                                                             "aplazamiento"],
        "Política de gratuidad (matrícula cero) Pregrado": ["matrícula cero", "gratuidad", "matrícula 0",
                                                             "política de gratuidad"],
        "Información general sobre servicios estudiantiles":["información general", "información financiera",
                                                              "servicios estudiantiles", "bienestar",
                                                              "alimentaria", "correo institucional",
                                                              "sia", "bicirún", "sibu"],
    }

    categoria_lower = categoria.lower()

    for cat_valida, palabras_clave in mapeo.items():
        for palabra in palabras_clave:
            if palabra in categoria_lower:
                return cat_valida

    # Si no hay coincidencia, intentar por la consulta (fallback)
    return "Información general sobre servicios estudiantiles"


# Aplicar el mapeo
df["Categoria_Mapeada"] = df["Categoria"].apply(mapear_categoria)

print("\n📊 Distribución de categorías mapeadas:")
print(df["Categoria_Mapeada"].value_counts())


📊 Distribución de categorías mapeadas:
Categoria_Mapeada
Gestión Económica                                    15751
Gestión Académica                                    10520
Información general sobre servicios estudiantiles     4714
Certificados                                          2656
Actualización de datos personales                      768
Carnetización                                          458
Calendario académico                                   232
Reubicación socioeconómica en Pregrado                  49
Política de gratuidad (matrícula cero) Pregrado         20
Name: count, dtype: int64


## 5. PREPROCESAMIENTO DE TEXTO

In [13]:
STOPWORDS_ES = set(stopwords.words("spanish"))

# Stopwords adicionales específicas del dominio universitario
STOPWORDS_EXTRA = {
    "universidad", "nacional", "colombia", "unal", "sede", "bogotá",
    "favor", "gracias", "buenas", "buenos", "días", "tardes", "noches",
    "cordial", "saludo", "atentamente", "amablemente", "presente",
    "correo", "motivo", "solicito", "solicitud", "quisiera", "quiero",
    "necesito", "requiero", "agradezco", "agradecería", "muchas",
    "manera", "forma", "caso", "parte", "vez", "día", "semestre",
    "periodo", "académico", "estudiante", "programa", "curricular",
    "sia", "dninfoa", "portal", "plataforma", "sistema",
}

STOPWORDS_COMPLETO = STOPWORDS_ES.union(STOPWORDS_EXTRA)


def limpiar_texto(texto: str) -> str:
    """Limpieza básica: minúsculas, eliminar caracteres especiales y números."""
    texto = str(texto).lower()
    # Eliminar URLs
    texto = re.sub(r"http\S+|www\S+", " ", texto)
    # Eliminar correos electrónicos
    texto = re.sub(r"\S+@\S+", " ", texto)
    # Eliminar números y caracteres especiales, conservar letras y espacios
    texto = re.sub(r"[^a-záéíóúüñ\s]", " ", texto)
    # Eliminar espacios múltiples
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto


def eliminar_stopwords(texto: str) -> str:
    """Elimina stopwords del texto."""
    tokens = texto.split()
    tokens_filtrados = [t for t in tokens if t not in STOPWORDS_COMPLETO and len(t) > 2]
    return " ".join(tokens_filtrados)


def lematizar(texto: str) -> str:
    """Lematiza el texto usando spaCy."""
    doc = nlp(texto)
    lemas = [
        token.lemma_ if token.lemma_ else token.text
        for token in doc
        if not token.is_stop
        and not token.is_punct
        and len(token.lemma_ if token.lemma_ else token.text) > 2
    ]
    return " ".join(lemas)


def preprocesar(texto: str) -> str:
    """Pipeline completo: limpieza → stopwords → lematización."""
    texto = limpiar_texto(texto)
    texto = eliminar_stopwords(texto)
    texto = lematizar(texto)
    return texto


print("\n⚙️  Aplicando preprocesamiento (puede tardar unos minutos)...")
df["Consulta_Procesada"] = df["Consulta"].apply(preprocesar)

print("✅ Preprocesamiento completado.")
print("\nEjemplo de transformación:")
print(f"  ORIGINAL : {df['Consulta'].iloc[0][:100]}...")
print(f"  PROCESADO: {df['Consulta_Procesada'].iloc[0][:100]}...")


⚙️  Aplicando preprocesamiento (puede tardar unos minutos)...
✅ Preprocesamiento completado.

Ejemplo de transformación:
  ORIGINAL : Ayuda no se como inscribir asignaturas y nadie me explica....
  PROCESADO: ayuda inscribir asignatura explicar...


## 6. PREPARACIÓN DE FEATURES Y ETIQUETAS

In [14]:
# Eliminar filas con texto procesado vacío
df = df[df["Consulta_Procesada"].str.strip() != ""]
df = df.reset_index(drop=True)

X = df["Consulta_Procesada"]
y = df["Categoria_Mapeada"]

# Codificar etiquetas
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print(f"\n📦 Total de muestras para entrenamiento: {len(X)}")
print(f"🏷️  Clases: {list(le.classes_)}")


📦 Total de muestras para entrenamiento: 35150
🏷️  Clases: ['Actualización de datos personales', 'Calendario académico', 'Carnetización', 'Certificados', 'Gestión Académica', 'Gestión Económica', 'Información general sobre servicios estudiantiles', 'Política de gratuidad (matrícula cero) Pregrado', 'Reubicación socioeconómica en Pregrado']


## 7. VECTORIZACIÓN TF-IDF

In [17]:
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
    norm="l2",
)

# Vectorizador de conteos (para BernoulliNB)
count_vec = CountVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    binary=True,             # Presencia/ausencia para BernoulliNB
)

X_tfidf = tfidf.fit_transform(X)
X_count  = count_vec.fit_transform(X)

print(f"\n🔢 Matriz TF-IDF : {X_tfidf.shape}")
print(f"🔢 Matriz Conteo : {X_count.shape}")


🔢 Matriz TF-IDF : (35150, 5000)
🔢 Matriz Conteo : (35150, 5000)


## 8. DIVISIÓN TRAIN / TEST

In [18]:
X_train_tfidf, X_test_tfidf, y_train, y_test = train_test_split(
    X_tfidf, y_encoded,
    test_size=0.2,
    random_state= SEED,
    stratify=y_encoded,
)

X_train_count, X_test_count, _, _ = train_test_split(
    X_count, y_encoded,
    test_size=0.2,
    random_state= SEED,
    stratify=y_encoded,
)

print(f"\n📊 Train: {X_train_tfidf.shape[0]} muestras | "
      f"Test: {X_test_tfidf.shape[0]} muestras")


📊 Train: 28120 muestras | Test: 7030 muestras


## 9. ENTRENAMIENTO DEL MODELO 

In [19]:
print("\n📐 Entrenando variantes de Naive Bayes...")

# 1. MultinomialNB
modelo_mnb = MultinomialNB(alpha=0.1)   # alpha: suavizado de Laplace
modelo_mnb.fit(X_train_tfidf, y_train)
print("  ✅ MultinomialNB entrenado.")

# 2. ComplementNB — recomendado para datasets desbalanceados
modelo_cnb = ComplementNB(alpha=0.1)
modelo_cnb.fit(X_train_tfidf, y_train)
print("  ✅ ComplementNB entrenado.")

# 3. BernoulliNB — usa matriz binaria de presencia/ausencia
modelo_bnb = BernoulliNB(alpha=0.1)
modelo_bnb.fit(X_train_count, y_train)
print("  ✅ BernoulliNB entrenado.")


📐 Entrenando variantes de Naive Bayes...
  ✅ MultinomialNB entrenado.
  ✅ ComplementNB entrenado.
  ✅ BernoulliNB entrenado.


COMPARACIÓN DE VARIANTES

In [20]:
print("\n🔬 Comparación de variantes Naive Bayes (CV 5-fold):")
print("=" * 65)
print(f"{'Variante':<20} {'CV Accuracy':>12} {'Std':>8} "
      f"{'Test Acc':>10} {'F1-W':>8}")
print("-" * 65)

variantes_info = [
    ("MultinomialNB",  modelo_mnb, X_tfidf,  X_test_tfidf),
    ("ComplementNB",   modelo_cnb, X_tfidf,  X_test_tfidf),
    ("BernoulliNB",    modelo_bnb, X_count,  X_test_count),
]

resultados_variantes = {}

for nombre, modelo_var, X_full, X_test_var in variantes_info:
    # Validación cruzada
    cv_scores = cross_val_score(
        modelo_var, X_full, y_encoded,
        cv=5, scoring="accuracy", n_jobs=-1
    )
    # Métricas en test
    y_pred_var  = modelo_var.predict(X_test_var)
    acc_test    = accuracy_score(y_test, y_pred_var)
    f1_weighted = f1_score(y_test, y_pred_var,
                           average="weighted", zero_division=0)

    resultados_variantes[nombre] = {
        "cv_mean": cv_scores.mean(),
        "cv_std":  cv_scores.std(),
        "acc":     acc_test,
        "f1_w":    f1_weighted,
    }

    print(f"{nombre:<20} {cv_scores.mean():>12.4f} {cv_scores.std():>8.4f} "
          f"{acc_test:>10.4f} {f1_weighted:>8.4f}")

print("=" * 65)

# Seleccionar la mejor variante automáticamente
mejor_variante = max(
    resultados_variantes,
    key=lambda k: resultados_variantes[k]["f1_w"]
)
print(f"\n🏆 Mejor variante por F1-weighted: {mejor_variante}")


🔬 Comparación de variantes Naive Bayes (CV 5-fold):
Variante              CV Accuracy      Std   Test Acc     F1-W
-----------------------------------------------------------------
MultinomialNB              0.7380   0.0065     0.7499   0.7359
ComplementNB               0.7437   0.0060     0.7501   0.7347
BernoulliNB                0.6971   0.0163     0.7057   0.7107

🏆 Mejor variante por F1-weighted: MultinomialNB


## 10. EVALUACIÓN

In [21]:
# Mapeamos la mejor variante a su modelo y datos correspondientes
mapa_modelos = {
    "MultinomialNB": (modelo_mnb, X_test_tfidf, X_tfidf),
    "ComplementNB":  (modelo_cnb, X_test_tfidf, X_tfidf),
    "BernoulliNB":   (modelo_bnb, X_test_count, X_count),
}

modelo_final, X_test_final, X_full_final = mapa_modelos[mejor_variante]
y_pred = modelo_final.predict(X_test_final)

# ── Métricas globales ────────────────────────────────────────
accuracy        = accuracy_score(y_test, y_pred)
recall_macro    = recall_score(y_test, y_pred, average="macro",    zero_division=0)
recall_weighted = recall_score(y_test, y_pred, average="weighted", zero_division=0)
f1_macro        = f1_score(y_test, y_pred, average="macro",        zero_division=0)
f1_weighted     = f1_score(y_test, y_pred, average="weighted",     zero_division=0)
prec_macro      = precision_score(y_test, y_pred, average="macro",    zero_division=0)
prec_weighted   = precision_score(y_test, y_pred, average="weighted", zero_division=0)

print(f"\n{'=' * 55}")
print(f"   📊 MÉTRICAS GLOBALES — {mejor_variante}")
print(f"{'=' * 55}")
print(f"  🎯 Accuracy                  : {accuracy:.4f}  ({accuracy*100:.2f}%)")
print(f"  🔁 Recall    (macro)         : {recall_macro:.4f}  ({recall_macro*100:.2f}%)")
print(f"  🔁 Recall    (weighted)      : {recall_weighted:.4f}  ({recall_weighted*100:.2f}%)")
print(f"  📐 F1-Score  (macro)         : {f1_macro:.4f}  ({f1_macro*100:.2f}%)")
print(f"  📐 F1-Score  (weighted)      : {f1_weighted:.4f}  ({f1_weighted*100:.2f}%)")
print(f"  🎯 Precision (macro)         : {prec_macro:.4f}  ({prec_macro*100:.2f}%)")
print(f"  🎯 Precision (weighted)      : {prec_weighted:.4f}  ({prec_weighted*100:.2f}%)")
print(f"{'=' * 55}")

print("\n📋 Reporte de Clasificación por Clase:")
print(classification_report(
    y_test, y_pred,
    target_names=le.classes_,
    zero_division=0,
))

print("\n🔲 Matriz de Confusión:")
cm     = confusion_matrix(y_test, y_pred)
cm_df  = pd.DataFrame(cm, index=le.classes_, columns=le.classes_)
print(cm_df)


   📊 MÉTRICAS GLOBALES — MultinomialNB
  🎯 Accuracy                  : 0.7499  (74.99%)
  🔁 Recall    (macro)         : 0.4488  (44.88%)
  🔁 Recall    (weighted)      : 0.7499  (74.99%)
  📐 F1-Score  (macro)         : 0.4704  (47.04%)
  📐 F1-Score  (weighted)      : 0.7359  (73.59%)
  🎯 Precision (macro)         : 0.5395  (53.95%)
  🎯 Precision (weighted)      : 0.7352  (73.52%)

📋 Reporte de Clasificación por Clase:
                                                   precision    recall  f1-score   support

                Actualización de datos personales       0.79      0.47      0.59       153
                             Calendario académico       0.33      0.02      0.04        46
                                    Carnetización       0.90      0.75      0.82        92
                                     Certificados       0.75      0.81      0.78       531
                                Gestión Académica       0.71      0.72      0.71      2102
                               

## 11. ANÁLISIS DE VECTORES DE SOPORTE POR CLASE

In [22]:
# Característica exclusiva de Naive Bayes: log-probabilidades
# de cada término dado cada clase
print("\n🔑 Top 15 términos más relevantes por categoría (log-prob NB):")
print("=" * 65)

feature_names = tfidf.get_feature_names_out()

# Usar MultinomialNB o ComplementNB (tienen feature_log_prob_)
modelo_analisis = modelo_mnb if mejor_variante != "BernoulliNB" else modelo_bnb
fn_analisis     = (feature_names if mejor_variante != "BernoulliNB"
                   else count_vec.get_feature_names_out())

for i, clase in enumerate(le.classes_):
    log_probs   = modelo_analisis.feature_log_prob_[i]
    top_indices = np.argsort(log_probs)[::-1][:15]
    top_terms   = [(fn_analisis[j], round(log_probs[j], 3))
                   for j in top_indices]
    print(f"\n🏷️  {clase}")
    print(f"   {top_terms}")

print("=" * 65)


🔑 Top 15 términos más relevantes por categoría (log-prob NB):

🏷️  Actualización de datos personales
   [('dato', np.float64(-4.313)), ('actualización', np.float64(-4.332)), ('actualizar', np.float64(-4.419)), ('documento', np.float64(-4.554)), ('formulario', np.float64(-4.597)), ('identidad', np.float64(-4.617)), ('cambio', np.float64(-4.659)), ('tipo documento', np.float64(-4.842)), ('tipo', np.float64(-4.968)), ('actualización dato', np.float64(-4.972)), ('cambiar', np.float64(-5.07)), ('dato personal', np.float64(-5.179)), ('permiso', np.float64(-5.255)), ('personal', np.float64(-5.257)), ('documento identidad', np.float64(-5.26))]

🏷️  Calendario académico
   [('calendario', np.float64(-4.28)), ('fecha', np.float64(-4.542)), ('clase', np.float64(-5.059)), ('información', np.float64(-5.302)), ('inscripción', np.float64(-5.34)), ('admitido', np.float64(-5.34)), ('inducción', np.float64(-5.404)), ('proceso', np.float64(-5.491)), ('semana', np.float64(-5.504)), ('matrícula', np.float

## 12. EFECTO DEL SUAVIZADO ALPHA

In [23]:
print("\n🔧 Efecto del parámetro Alpha (suavizado de Laplace):")
print("=" * 55)
print(f"{'Alpha':<10} {'Accuracy':>10} {'F1-Weighted':>13} {'CV Mean':>10}")
print("-" * 55)

alphas = [0.01, 0.05, 0.1, 0.5, 1.0, 2.0, 5.0]

for alpha_val in alphas:
    mnb_tmp = ComplementNB(alpha=alpha_val)
    mnb_tmp.fit(X_train_tfidf, y_train)
    y_tmp      = mnb_tmp.predict(X_test_tfidf)
    acc_tmp    = accuracy_score(y_test, y_tmp)
    f1_tmp     = f1_score(y_test, y_tmp, average="weighted", zero_division=0)
    cv_tmp     = cross_val_score(
        mnb_tmp, X_tfidf, y_encoded, cv=5, scoring="accuracy"
    ).mean()
    print(f"{alpha_val:<10} {acc_tmp:>10.4f} {f1_tmp:>13.4f} {cv_tmp:>10.4f}")

print("=" * 55)


🔧 Efecto del parámetro Alpha (suavizado de Laplace):
Alpha        Accuracy   F1-Weighted    CV Mean
-------------------------------------------------------
0.01           0.7499        0.7345     0.7434
0.05           0.7501        0.7346     0.7437
0.1            0.7501        0.7347     0.7437
0.5            0.7498        0.7344     0.7440
1.0            0.7496        0.7345     0.7435
2.0            0.7491        0.7338     0.7434
5.0            0.7536        0.7370     0.7435


## 13. FUNCIÓN DE PREDICCIÓN

In [24]:
def predecir_categoria(texto_nuevo: str) -> dict:
    """
    Predice la categoría de una nueva consulta con Naive Bayes.
    Retorna categoría predicha, confianza y probabilidades por clase.
    """
    texto_proc = preprocesar(texto_nuevo)

    # Seleccionar vectorizador según la mejor variante
    if mejor_variante == "BernoulliNB":
        texto_vec = count_vec.transform([texto_proc])
    else:
        texto_vec = tfidf.transform([texto_proc])

    pred_encoded   = modelo_final.predict(texto_vec)[0]
    pred_categoria = le.inverse_transform([pred_encoded])[0]
    log_probs      = modelo_final.predict_log_proba(texto_vec)[0]
    probabilidades = modelo_final.predict_proba(texto_vec)[0]

    probs_dict = {
        clase: round(float(prob), 4)
        for clase, prob in zip(le.classes_, probabilidades)
    }
    probs_ordenadas = dict(
        sorted(probs_dict.items(), key=lambda x: x[1], reverse=True)
    )

    confianza = float(max(probabilidades))

    return {
        "categoria_predicha": pred_categoria,
        "confianza":          round(confianza, 4),
        "probabilidades":     probs_ordenadas,
    }


## 14. PRUEBA CON EJEMPLOS

In [25]:
ejemplos = [
    "No me aparece el recibo de pago en el SIA y necesito pagarlo urgente",
    "Quisiera saber cómo inscribir materias para este semestre",
    "Necesito tramitar mi carné universitario",
    "Soy estrato 2 y quiero saber si aplico para matrícula cero",
    "Solicito certificado de notas para trámite externo",
    "Quiero aplazar mi matrícula por motivos económicos",
    "Mi PBM quedó muy alto y no tengo recursos para pagar",
    "No me asignaron cita de inscripción de asignaturas",
    "Necesito actualizar mi dirección y datos de contacto",
    "No sé cuándo empiezan las clases este semestre",
]

print("\n🧪 PRUEBAS DE PREDICCIÓN:")
print("=" * 68)
for ejemplo in ejemplos:
    resultado = predecir_categoria(ejemplo)
    print(f"\n📝 Consulta   : {ejemplo}")
    print(f"🏷️  Predicción : {resultado['categoria_predicha']}")
    print(f"🔒 Confianza  : {resultado['confianza']*100:.1f}%")
    top3 = list(resultado["probabilidades"].items())[:3]
    print(f"📊 Top-3 probs: {top3}")
print("=" * 68)


🧪 PRUEBAS DE PREDICCIÓN:

📝 Consulta   : No me aparece el recibo de pago en el SIA y necesito pagarlo urgente
🏷️  Predicción : Gestión Económica
🔒 Confianza  : 98.0%
📊 Top-3 probs: [('Gestión Económica', 0.9803), ('Gestión Académica', 0.016), ('Información general sobre servicios estudiantiles', 0.0032)]

📝 Consulta   : Quisiera saber cómo inscribir materias para este semestre
🏷️  Predicción : Gestión Académica
🔒 Confianza  : 85.2%
📊 Top-3 probs: [('Gestión Académica', 0.8517), ('Información general sobre servicios estudiantiles', 0.0896), ('Gestión Económica', 0.0531)]

📝 Consulta   : Necesito tramitar mi carné universitario
🏷️  Predicción : Carnetización
🔒 Confianza  : 79.7%
📊 Top-3 probs: [('Carnetización', 0.7973), ('Información general sobre servicios estudiantiles', 0.0837), ('Gestión Económica', 0.0542)]

📝 Consulta   : Soy estrato 2 y quiero saber si aplico para matrícula cero
🏷️  Predicción : Gestión Económica
🔒 Confianza  : 63.8%
📊 Top-3 probs: [('Gestión Económica', 0.6381)

## 15. GUARDAR RESULTADOS

In [26]:
y_all_pred  = modelo_final.predict(X_full_final)
y_all_proba = modelo_final.predict_proba(X_full_final)

df_resultado = df[[
    "Consulta", "Categoria", "Categoria_Mapeada", "Consulta_Procesada"
]].copy()

df_resultado["Prediccion"]      = le.inverse_transform(y_all_pred)
df_resultado["Correcto"]        = (
    df_resultado["Categoria_Mapeada"] == df_resultado["Prediccion"]
)
df_resultado["Confianza"]       = np.max(y_all_proba, axis=1).round(4)

# Tabla de métricas por clase
reporte_dict = classification_report(
    y_test, y_pred,
    target_names=le.classes_,
    zero_division=0,
    output_dict=True,
)
df_metricas = pd.DataFrame(reporte_dict).T.round(4)

# Tabla comparativa de variantes
df_variantes = pd.DataFrame(resultados_variantes).T.round(4)

with pd.ExcelWriter("resultados_naive_bayes.xlsx", engine="openpyxl") as writer:
    df_resultado.to_excel(writer, sheet_name="Predicciones",      index=False)
    df_metricas.to_excel(writer,  sheet_name="Metricas_por_clase")
    df_variantes.to_excel(writer, sheet_name="Comparacion_variantes")

print("\n💾 Archivo guardado:")
print("   → resultados_naive_bayes.xlsx")
print("      • Hoja 1: Predicciones")
print("      • Hoja 2: Métricas por clase")
print("      • Hoja 3: Comparación de variantes")


💾 Archivo guardado:
   → resultados_naive_bayes.xlsx
      • Hoja 1: Predicciones
      • Hoja 2: Métricas por clase
      • Hoja 3: Comparación de variantes
